<a href="https://colab.research.google.com/github/m1Febriansyah/244107020199-ML-12/blob/main/JS03_Tugas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

# 1. Memuat Data dan Pemisahan Variabel
df = pd.read_csv("wbc.csv")
kolom_tidak_terpakai = ['id', 'Unnamed: 32']
df_berguna = df.drop(columns=kolom_tidak_terpakai)

# 2. Encoding pada kolom diagnosis
le = LabelEncoder()
df_berguna['diagnosis'] = le.fit_transform(df_berguna['diagnosis'])

# Memisahkan Fitur (X) dan Target (y)
X = df_berguna.drop('diagnosis', axis=1)
y = df_berguna['diagnosis']

# Membagi data latih dan data uji (80:20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. Membuat Pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),                 # Standardisasi Numerik
    ('selector', SelectKBest(score_func=f_classif)), # Seleksi Fitur
    ('classifier', LogisticRegression(random_state=42)) # Model
])

# 4. Mencari jumlah fitur terbaik menggunakan GridSearchCV
# Menguji jumlah K dari 1 sampai 30 fitur
param_grid = {'selector__k': list(range(1, 31))}

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

# Menampilkan hasil model terbaik
best_k = grid_search.best_params_['selector__k']
print(f"Jumlah K terbaik: {best_k}")
print(f"Akurasi Validasi (CV): {grid_search.best_score_ * 100:.2f}%")

# Menguji pada data uji (Test Set)
y_pred = grid_search.best_estimator_.predict(X_test)
print(f"Akurasi Data Uji: {accuracy_score(y_test, y_pred) * 100:.2f}%")

# 5. Menampilkan Fitur Apa Saja yang Terpilih
# Mengambil nama fitur dari estimator terbaik di dalam pipeline
mask_fitur_terpilih = grid_search.best_estimator_.named_steps['selector'].get_support()
fitur_terpilih = X.columns[mask_fitur_terpilih].tolist()

print(f"\nDaftar {best_k} Fitur Terbaik:")
for i, fitur in enumerate(fitur_terpilih, 1):
    print(f"{i}. {fitur}")

Jumlah K terbaik: 19
Akurasi Validasi (CV): 97.36%
Akurasi Data Uji: 98.25%

Daftar 19 Fitur Terbaik:
1. radius_mean
2. texture_mean
3. perimeter_mean
4. area_mean
5. compactness_mean
6. concavity_mean
7. concave points_mean
8. radius_se
9. perimeter_se
10. area_se
11. radius_worst
12. texture_worst
13. perimeter_worst
14. area_worst
15. smoothness_worst
16. compactness_worst
17. concavity_worst
18. concave points_worst
19. symmetry_worst
